In [ ]:
# This mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive')

FOLDERNAME = 'intro_to_ai/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load

# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the CIFAR-10 dataset to your Drive
# if it doesn't already exist.
%cd /content/drive/My\ Drive/$FOLDERNAME/intro_to_ai/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

Mounted at /content/drive
[Errno 2] No such file or directory: '/content/drive/My Drive/intro_to_ai/assignments/assignment1//intro_to_ai/datasets/'
/content
bash: get_datasets.sh: No such file or directory
[Errno 2] No such file or directory: '/content/drive/My Drive/intro_to_ai/assignments/assignment1/'
/content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Table of Contents

This assignment has 5 parts. You will learn PyTorch on **three different levels of abstraction**, which will help you understand it better and prepare you for the final project.

1. Part I, Preparation: we will use CIFAR-10 dataset.
2. Part II, Barebones PyTorch: **Abstraction level 1**, we will work directly with the lowest-level PyTorch Tensors.
3. Part III, PyTorch Module API: **Abstraction level 2**, we will use `nn.Module` to define arbitrary neural network architecture.
4. Part IV, PyTorch Sequential API: **Abstraction level 3**, we will use `nn.Sequential` to define a linear feed-forward network very conveniently.
5. Part V, CIFAR-10 open-ended challenge: please implement your own network to get as high accuracy as possible on CIFAR-10. You can experiment with any layer, optimizer, hyperparameters or other advanced features.

Here is a table of comparison:

| API           | Flexibility | Convenience |
|---------------|-------------|-------------|
| Barebone      | High        | Low         |
| `nn.Module`     | High        | Medium      |
| `nn.Sequential` | Low         | High        |

# GPU

You can manually switch to a GPU device on Colab by clicking `Runtime -> Change runtime type` and selecting `GPU` under `Hardware Accelerator`. You should do this before running the following cells to import packages, since the kernel gets restarted upon switching runtimes.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this assignment.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)

using device: cuda


# Part I. Preparation

Now, let's load the CIFAR-10 dataset. This might take a couple of minutes the first
time you run the code, but the files should stay cached afterwards.

In earlier sections of this assignment, dataset handling was performed manually.
Here, we use PyTorch's built-in utilities to load, preprocess, and iterate over
the data in minibatches.


In [ ]:
NUM_TRAIN = 49000

# The torchvision.transforms package provides tools for preprocessing data.
# Here we set up a transform that normalizes the data by subtracting the mean
# RGB value and dividing by the standard deviation of each RGB channel.
# The mean and standard deviation values are hardcoded.
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2023, 0.1994, 0.2010))
            ])

# We set up a Dataset object for each split (train / val / test).
# Datasets load examples one at a time, so we wrap each Dataset in a DataLoader
# which forms minibatches. The training set is split into train and validation
# subsets using a Sampler.
cifar10_train = dset.CIFAR10('./intro_to_ai/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./intro_to_ai/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./intro_to_ai/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)


100%|██████████| 170M/170M [00:13<00:00, 12.6MB/s]


# Part II. Barebones PyTorch

PyTorch ships with high-level APIs to help us define model architectures conveniently, which we will cover in Part II of this assignment. In this section, we will start with the barebone PyTorch elements to understand the autograd engine better. After this exercise, you will come to appreciate the high-level model API more.

We will start with a simple fully-connected ReLU network with two hidden layers and no biases for CIFAR classification.
This implementation computes the forward pass using operations on PyTorch Tensors, and uses PyTorch autograd to compute gradients. It is important that you understand every line, because you will write a harder version after the example.

When we create a PyTorch Tensor with `requires_grad=True`, then operations involving that Tensor will not just compute values; they will also build up a computational graph in the background, allowing us to easily backpropagate through the graph to compute gradients of some Tensors with respect to a downstream loss. Concretely if x is a Tensor with `x.requires_grad == True` then after backpropagation `x.grad` will be another Tensor holding the gradient of x with respect to the scalar loss at the end.

### PyTorch Tensors: Flatten Function
A PyTorch Tensor is conceptionally similar to a numpy array: it is an n-dimensional grid of numbers, and like numpy PyTorch provides many functions to efficiently operate on Tensors. As a simple example, we provide a `flatten` function below which reshapes image data for use in a fully-connected neural network.

Recall that image data is typically stored in a Tensor of shape N x C x H x W, where:

* N is the number of datapoints
* C is the number of channels
* H is the height of the intermediate feature map in pixels
* W is the height of the intermediate feature map in pixels

When we use fully connected affine layers to process the image, however, we want each datapoint to be represented by a single vector -- it's no longer useful to segregate the different channels, rows, and columns of the data. So, we use a "flatten" operation to collapse the `C x H x W` values per representation into a single long vector. The flatten function below first reads in the N, C, H, and W values from a given batch of data, and then returns a "view" of that data. "View" is analogous to numpy's "reshape" method: it reshapes x's dimensions to be N x ??, where ?? is allowed to be anything (in this case, it will be C x H x W, but we don't need to specify that explicitly).

In [ ]:
def flatten(x):
    N = x.shape[0] # read in N, C, H, W
    return x.view(N, -1)  # "flatten" the C * H * W values into a single vector per image

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()

Before flattening:  tensor([[[[ 0,  1],
          [ 2,  3],
          [ 4,  5]]],


        [[[ 6,  7],
          [ 8,  9],
          [10, 11]]]])
After flattening:  tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


### Barebones PyTorch: Two-Layer Network

Here we define a function `two_layer_fc` which performs the forward pass of a two-layer fully-connected ReLU network on a batch of image data. After defining the forward pass we check that it doesn't crash and that it produces outputs of the right shape by running zeros through the network.

You don't have to write any code here, but it's important that you read and understand the implementation.

In [ ]:
import torch.nn.functional as F  # useful stateless functions

def two_layer_fc(x, params):
    """
    A fully-connected neural network; the architecture is:
    NN is fully connected -> ReLU -> fully connected layer.
    Note that this function only defines the forward pass;
    PyTorch will take care of the backward pass for us.

    The input to the network will be a minibatch of data, of shape
    (N, d1, ..., dM) where d1 * ... * dM = D. The hidden layer will have H units,
    and the output layer will produce scores for C classes.

    Inputs:
    - x: A PyTorch Tensor of shape (N, d1, ..., dM) giving a minibatch of
      input data.
    - params: A list [w1, w2] of PyTorch Tensors giving weights for the network;
      w1 has shape (D, H) and w2 has shape (H, C).

    Returns:
    - scores: A PyTorch Tensor of shape (N, C) giving classification scores for
      the input data x.
    """
    # first we flatten the image
    x = flatten(x)  # shape: [batch_size, C x H x W]

    w1, w2 = params

    # Forward pass: compute predicted y using operations on Tensors. Since w1 and
    # w2 have requires_grad=True, operations involving these Tensors will cause
    # PyTorch to build a computational graph, allowing automatic computation of
    # gradients. Since we are no longer implementing the backward pass by hand we
    # don't need to keep references to intermediate values.
    # you can also use `.clamp(min=0)`, equivalent to F.relu()
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x


def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)  # minibatch size 64, feature dimension 50
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())  # you should see [64, 10]

two_layer_fc_test()

torch.Size([64, 10])


### Barebones PyTorch: Initialization
Let's write a couple utility methods to initialize the weight matrices for our models.

- `random_weight(shape)` initializes a weight tensor with the Kaiming normalization method.
- `zero_weight(shape)` initializes a weight tensor with all zeros. Useful for instantiating bias parameters.

The `random_weight` function uses the Kaiming normal initialization method, described in:

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852

In [ ]:
def random_weight(shape):
    """
    Create random Tensors for weights; setting requires_grad=True means that we
    want to compute gradients for these Tensors during the backward pass.
    We use Kaiming normalization: sqrt(2 / fan_in)
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # general fan-in computation
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# create a weight of shape [3 x 5]
# you should see the type `torch.cuda.FloatTensor` if you use GPU.
# Otherwise it should be `torch.FloatTensor`
random_weight((3, 5))

tensor([[-0.2956, -0.4053, -1.9537, -0.4938, -0.0732],
        [ 0.2454, -0.9170,  0.2676,  0.6918, -0.3692],
        [-1.2002, -0.9966, -2.2501,  0.9938, -0.0074]], device='cuda:0',
       requires_grad=True)

### Barebones PyTorch: Check Accuracy
When training the model we will use the following function to check the accuracy of our model on the training or validation sets.

When checking accuracy we don't need to compute any gradients; as a result we don't need PyTorch to build a computational graph for us when we compute scores. To prevent a graph from being built we scope our computation under a `torch.no_grad()` context manager.

In [ ]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model_fn: A function that performs the forward pass of the model,
      with the signature scores = model_fn(x, params)
    - params: List of PyTorch Tensors giving parameters of the model

    Returns: Nothing, but prints the accuracy of the model
    """
    split = 'val' if loader.dataset.train else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

### BareBones PyTorch: Training Loop
We can now set up a basic training loop to train our network. We will train the model using stochastic gradient descent without momentum. We will use `torch.functional.cross_entropy` to compute the loss; you can [read about it here](http://pytorch.org/docs/stable/nn.html#cross-entropy).

The training loop takes as input the neural network function, a list of initialized parameters (`[w1, w2]` in our example), and learning rate.

In [ ]:
def train_part2(model_fn, params, learning_rate):
    """
    Train a model on CIFAR-10.

    Inputs:
    - model_fn: A Python function that performs the forward pass of the model.
      It should have the signature scores = model_fn(x, params) where x is a
      PyTorch Tensor of image data, params is a list of PyTorch Tensors giving
      model weights, and scores is a PyTorch Tensor of shape (N, C) giving
      scores for the elements in x.
    - params: List of PyTorch Tensors giving weights for the model
    - learning_rate: Python scalar giving the learning rate to use for SGD

    Returns: Nothing
    """
    for t, (x, y) in enumerate(loader_train):
        # Move the data to the proper device (GPU or CPU)
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # Forward pass: compute scores and loss
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # Backward pass: PyTorch figures out which Tensors in the computational
        # graph has requires_grad=True and uses backpropagation to compute the
        # gradient of the loss with respect to these Tensors, and stores the
        # gradients in the .grad attribute of each Tensor.
        loss.backward()

        # Update parameters. We don't want to backpropagate through the
        # parameter updates, so we scope the updates under a torch.no_grad()
        # context manager to prevent a computational graph from being built.
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # Manually zero the gradients after running the backward pass
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()

### BareBones PyTorch: Train a Two-Layer Network
Now we are ready to run the training loop. We need to explicitly allocate tensors for the fully connected weights, `w1` and `w2`.

Each minibatch of CIFAR has 64 examples, so the tensor shape is `[64, 3, 32, 32]`.

After flattening, `x` shape should be `[64, 3 * 32 * 32]`. This will be the size of the first dimension of `w1`.
The second dimension of `w1` is the hidden layer size, which will also be the first dimension of `w2`.

Finally, the output of the network is a 10-dimensional vector that represents the probability distribution over 10 classes.

You do not need to tune any hyperparameters. After training for one epoch, you
should observe that the model learns non-trivial structure in the data.


In [ ]:
hidden_layer_size = 4000
learning_rate = 1e-2

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)

Iteration 0, loss = 3.5967
Checking accuracy on the val set
Got 177 / 1000 correct (17.70%)

Iteration 100, loss = 2.7104
Checking accuracy on the val set
Got 288 / 1000 correct (28.80%)

Iteration 200, loss = 2.5225
Checking accuracy on the val set
Got 335 / 1000 correct (33.50%)

Iteration 300, loss = 1.9980
Checking accuracy on the val set
Got 347 / 1000 correct (34.70%)

Iteration 400, loss = 1.7257
Checking accuracy on the val set
Got 441 / 1000 correct (44.10%)

Iteration 500, loss = 1.7779
Checking accuracy on the val set
Got 448 / 1000 correct (44.80%)

Iteration 600, loss = 1.6159
Checking accuracy on the val set
Got 457 / 1000 correct (45.70%)

Iteration 700, loss = 1.6328
Checking accuracy on the val set
Got 436 / 1000 correct (43.60%)



# Part III. PyTorch Module API

Barebone PyTorch requires that we track all the parameter tensors by hand. This is fine for small networks with a few tensors, but it would be extremely inconvenient and error-prone to track tens or hundreds of tensors in larger networks.

PyTorch provides the `nn.Module` API for you to define arbitrary network architectures, while tracking every learnable parameters for you. In Part II, we implemented SGD ourselves. PyTorch also provides the `torch.optim` package that implements all the common optimizers, such as RMSProp, Adagrad, and Adam. It even supports approximate second-order methods like L-BFGS! You can refer to the [doc](http://pytorch.org/docs/master/optim.html) for the exact specifications of each optimizer.

To use the Module API, follow the steps below:

1. Subclass `nn.Module`. Give your network class an intuitive name like `TwoLayerFC`.

2. In the constructor `__init__()`, define all the layers you need as class attributes. Layer objects like `nn.Linear` are themselves `nn.Module` subclasses and contain learnable parameters, so that you don't have to instantiate the raw tensors yourself. `nn.Module` will track these internal parameters for you. Refer to the [doc](http://pytorch.org/docs/master/nn.html) to learn more about the dozens of builtin layers. **Warning**: don't forget to call the `super().__init__()` first!

3. In the `forward()` method, define the *connectivity* of your network. You should use the attributes defined in `__init__` as function calls that take tensor as input and output the "transformed" tensor. Do *not* create any new layers with learnable parameters in `forward()`! All of them must be declared upfront in `__init__`.

After you define your Module subclass, you can instantiate it as an object and call it just like the NN forward function in part II.

### Module API: Two-Layer Network
Here is a concrete example of a 2-layer fully connected network:

In [ ]:
class TwoLayerFC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # assign layer objects to class attributes
        self.fc1 = nn.Linear(input_size, hidden_size)
        # nn.init package contains convenient initialization methods
        # http://pytorch.org/docs/master/nn.html#torch-nn-init
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        # forward always defines connectivity
        x = flatten(x)
        scores = self.fc2(F.relu(self.fc1(x)))
        return scores

def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)  # minibatch size 64, feature dimension 50
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())  # you should see [64, 10]
test_TwoLayerFC()

torch.Size([64, 10])


### Module API: Check Accuracy
Given the validation or test set, we can check the classification accuracy of a neural network.

This version is slightly different from the one in part II. You don't manually pass in the parameters anymore.

In [ ]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

### Module API: Training Loop
We also use a slightly different training loop. Rather than updating the values of the weights ourselves, we use an Optimizer object from the `torch.optim` package, which abstract the notion of an optimization algorithm and provides implementations of most of the algorithms commonly used to optimize neural networks.

In [ ]:
def train_part34(model, optimizer, epochs=1):
    """
    Train a model on CIFAR-10 using the PyTorch Module API.

    Inputs:
    - model: A PyTorch Module giving the model to train.
    - optimizer: An Optimizer object we will use to train the model
    - epochs: (Optional) A Python integer giving the number of epochs to train for

    Returns: Nothing, but prints model accuracies during training.
    """
    model = model.to(device=device)  # move the model parameters to CPU/GPU
    for e in range(epochs):
        for t, (x, y) in enumerate(loader_train):
            model.train()  # put model to training mode
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            # Zero out all of the gradients for the variables which the optimizer
            # will update.
            optimizer.zero_grad()

            # This is the backwards pass: compute the gradient of the loss with
            # respect to each  parameter of the model.
            loss.backward()

            # Actually update the parameters of the model using the gradients
            # computed by the backwards pass.
            optimizer.step()

            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                check_accuracy_part34(loader_val, model)
                print()

### Module API: Train a Two-Layer Network
Now we are ready to run the training loop. In contrast to part II, we don't explicitly allocate parameter tensors anymore.

Simply pass the input size, hidden layer size, and number of classes (i.e. output size) to the constructor of `TwoLayerFC`.

You also need to define an optimizer that tracks all the learnable parameters inside `TwoLayerFC`.

You do not need to tune any hyperparameters. After training for one epoch, you
should observe that the model learns non-trivial structure in the data.


In [ ]:
hidden_layer_size = 4000
learning_rate = 1e-2
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

train_part34(model, optimizer)

Iteration 0, loss = 3.8247
Checking accuracy on validation set
Got 150 / 1000 correct (15.00)

Iteration 100, loss = 2.0783
Checking accuracy on validation set
Got 347 / 1000 correct (34.70)

Iteration 200, loss = 1.9491
Checking accuracy on validation set
Got 360 / 1000 correct (36.00)

Iteration 300, loss = 2.0362
Checking accuracy on validation set
Got 426 / 1000 correct (42.60)

Iteration 400, loss = 2.1130
Checking accuracy on validation set
Got 411 / 1000 correct (41.10)

Iteration 500, loss = 1.9090
Checking accuracy on validation set
Got 434 / 1000 correct (43.40)

Iteration 600, loss = 1.7469
Checking accuracy on validation set
Got 408 / 1000 correct (40.80)

Iteration 700, loss = 1.8748
Checking accuracy on validation set
Got 440 / 1000 correct (44.00)



# Part IV. PyTorch Sequential API

Part III introduced the PyTorch Module API, which allows you to define arbitrary learnable layers and their connectivity.

For simple models like a stack of feed forward layers, you still need to go through 3 steps: subclass `nn.Module`, assign layers to class attributes in `__init__`, and call each layer one by one in `forward()`. Is there a more convenient way?

Fortunately, PyTorch provides a container Module called `nn.Sequential`, which merges the above steps into one. It is not as flexible as `nn.Module`, because you cannot specify more complex topology than a feed-forward stack, but it's good enough for many use cases.

### Sequential API: Two-Layer Network
Let's see how to rewrite our two-layer fully connected network example with `nn.Sequential`, and train it using the training loop defined above.

Again, you do not need to tune any hyperparameters. After one epoch of training,
you should observe that the model learns non-trivial structure in the data.


In [ ]:
# We need to wrap `flatten` function in a module in order to stack it
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 4000
learning_rate = 1e-2

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part34(model, optimizer)

Iteration 0, loss = 2.3265
Checking accuracy on validation set
Got 141 / 1000 correct (14.10)

Iteration 100, loss = 1.9521
Checking accuracy on validation set
Got 377 / 1000 correct (37.70)

Iteration 200, loss = 1.7901
Checking accuracy on validation set
Got 399 / 1000 correct (39.90)

Iteration 300, loss = 2.0879
Checking accuracy on validation set
Got 440 / 1000 correct (44.00)

Iteration 400, loss = 1.7750
Checking accuracy on validation set
Got 443 / 1000 correct (44.30)

Iteration 500, loss = 1.7914
Checking accuracy on validation set
Got 428 / 1000 correct (42.80)

Iteration 600, loss = 2.1436
Checking accuracy on validation set
Got 432 / 1000 correct (43.20)

Iteration 700, loss = 1.9150
Checking accuracy on validation set
Got 419 / 1000 correct (41.90)



# Part V. CIFAR-10 open-ended challenge

In this section, you can experiment with whatever architecture you'd like on CIFAR-10.

Now it's your job to experiment with architectures, hyperparameters, loss functions, and optimizers to train a model that achieves high accuracy on the CIFAR-10 **validation** set within 10 epochs. You can use the check_accuracy and train functions from above. You can use either `nn.Module` or `nn.Sequential` API.

Describe what you did at Overleaf report.

* Layers in torch.nn package: http://pytorch.org/docs/stable/nn.html
* Activations: http://pytorch.org/docs/stable/nn.html#non-linear-activations
* Loss functions: http://pytorch.org/docs/stable/nn.html#loss-functions
* Optimizers: http://pytorch.org/docs/stable/optim.html


### Things you might try:
- **Batch normalization**: Try adding batch normalization after fully-connected
  layers. Does training become more stable or faster?
- **Network architecture**: The network above has two layers of trainable parameters. Can you do better with a deep network?
- **Regularization**: Add l2 weight regularization, or perhaps use Dropout.

### Tips for training
For each network architecture that you try, you should tune the learning rate and other hyperparameters. When doing this there are a couple important things to keep in mind:

- If the parameters are working well, you should see improvement within a few hundred iterations
- Remember the coarse-to-fine approach for hyperparameter tuning: start by testing a large range of hyperparameters for just a few training iterations to find the combinations of parameters that are working at all.
- Once you have found some sets of parameters that seem to work, search more finely around these parameters. You may need to train for more epochs.
- You should use the validation set for hyperparameter search, and save your test set for evaluating your architecture on the best parameters as selected by the validation set.

### Going above and beyond
If you are feeling adventurous there are many other features you can implement to try and improve your performance. You are **not required** to implement any of these, but don't miss the fun if you have time!

- Alternative optimizers: you can try Adam, Adagrad, RMSprop, etc.
- Alternative activation functions such as leaky ReLU, parametric ReLU, ELU, or MaxOut.
- Model ensembles
- Data augmentation
- New Architectures
  - [ResNets](https://arxiv.org/abs/1512.03385) where the input from the previous layer is added to the output.
  - [DenseNets](https://arxiv.org/abs/1608.06993) where inputs into previous layers are concatenated together.
  - [This blog has an in-depth overview](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)
  - [LeNet-5](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf) where convolutional layers and pooling are used to exploit spatial structure in images.
  - [AlexNet](https://papers.nips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf) which demonstrated the effectiveness of deep convolutional neural networks for large-scale image classification.
  - [This blog provides an accessible introduction to CNNs](https://cs231n.github.io/convolutional-networks/) explaining convolution, pooling, and why CNNs outperform fully-connected networks on image data.


### Have fun and happy training!

In [ ]:
################################################################################
# TODO:                                                                        #
# Experiment with any architectures,optimizers, and hyperparameters.          #
# Achieve high accuracy on the *validation set* within 10 epochs.              #
################################################################################

import torchvision.transforms as T
import torchvision.datasets as dset
from torch.utils.data import DataLoader, sampler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# ── 1. Data Augmentation ─────────────────────────────────────────────────────
transform_train = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010))
])

cifar10_train_aug = dset.CIFAR10('./intro_to_ai/datasets', train=True,
                                  download=True, transform=transform_train)
loader_train_aug = DataLoader(cifar10_train_aug, batch_size=128,
                               sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

# NOTE: Need to define loader_val and NUM_TRAIN within this cell or ensure they are globally available.
# Assuming NUM_TRAIN and device/dtype are already defined in previous executed cells and accessible.
# Adding explicit definitions for robustness in this self-contained block.
NUM_TRAIN = 49000 # Assuming this constant is needed for the sampler

# Val/Test transform (assuming it's needed for loader_val)
transform_eval = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010))
])

cifar10_val = dset.CIFAR10('./intro_to_ai/datasets', train=True, download=True,
                           transform=transform_eval)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

# Ensure device and dtype are defined for this cell's execution
USE_GPU = True # Assuming this global variable is used
dtype = torch.float32 # Assuming this global variable is used
if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print_every = 100 # Assuming this constant is used in the training loop

# ── 2. Architecture with BatchNorm ───────────────────────────────────────────
model = nn.Sequential(
    nn.Flatten(),

    nn.Linear(3072, 2048),
    nn.BatchNorm1d(2048),
    nn.ReLU(),
    nn.Dropout(0.4),

    nn.Linear(2048, 1024),
    nn.BatchNorm1d(1024),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(1024, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(512, 10)
).to(device=device)

# ── 3. Optimizer + LR Scheduler ──────────────────────────────────────────────
learning_rate = 1e-3
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

################################################################################
#                            END OF YOUR CODE                                  #
################################################################################

# ── 4. Training loop ──────────────────────────────────────────────────────────
best_val_acc = 0.0
best_weights = None

for epoch in range(10):
    print(f"\n=== Epoch {epoch+1}/10 ===")
    model.train()

    for t, (x, y) in enumerate(loader_train_aug):
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        optimizer.zero_grad()
        scores = model(x)
        loss = F.cross_entropy(scores, y)
        loss.backward()
        optimizer.step()

        if t % print_every == 0:
            print(f'  Iteration {t}, loss = {loss.item():.4f}')

    # ── חישוב val accuracy בסוף כל epoch ──────────────────────────────────
    num_correct, num_samples = 0, 0
    model.eval()
    with torch.no_grad():
        for x, y in loader_val:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            _, preds = model(x).max(1)
            num_correct += (preds == y).sum().item()
            num_samples += y.size(0)
    val_acc = num_correct / num_samples
    print(f'  → Val accuracy: {val_acc*100:.2f}%')

    # ── שמירת המשקולות הטובות ביותר ───────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        print(f'  ★ New best saved: {val_acc*100:.2f}%')

    scheduler.step()

# ── 5. טעינת המודל הטוב ביותר ─────────────────────────────────────────────
print(f'\nBest validation accuracy across all epochs: {best_val_acc*100:.2f}%')
model.load_state_dict(best_weights)
best_model = model


=== Epoch 1/10 ===
  Iteration 0, loss = 2.3894
  Iteration 100, loss = 1.8540
  Iteration 200, loss = 1.9203
  Iteration 300, loss = 1.9063
  → Val accuracy: 40.20%
  ★ New best saved: 40.20%

=== Epoch 2/10 ===
  Iteration 0, loss = 1.9224
  Iteration 100, loss = 1.8189
  Iteration 200, loss = 1.8323
  Iteration 300, loss = 1.7091
  → Val accuracy: 44.90%
  ★ New best saved: 44.90%

=== Epoch 3/10 ===
  Iteration 0, loss = 1.8137
  Iteration 100, loss = 1.6127
  Iteration 200, loss = 1.6856
  Iteration 300, loss = 1.6796
  → Val accuracy: 48.30%
  ★ New best saved: 48.30%

=== Epoch 4/10 ===
  Iteration 0, loss = 1.6515
  Iteration 100, loss = 1.6986
  Iteration 200, loss = 1.7082
  Iteration 300, loss = 1.5638
  → Val accuracy: 49.10%
  ★ New best saved: 49.10%

=== Epoch 5/10 ===
  Iteration 0, loss = 1.5182
  Iteration 100, loss = 1.6251
  Iteration 200, loss = 1.6523
  Iteration 300, loss = 1.4474
  → Val accuracy: 47.80%

=== Epoch 6/10 ===
  Iteration 0, loss = 1.5947
  Iterat

In [1]:
################################################################################
# Part V: CNN — GPU Optimized version
################################################################################

!pip install optuna tqdm -q

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.datasets as dset
import torchvision.transforms as T
from torch.utils.data import DataLoader, sampler
from tqdm.auto import tqdm, trange
import numpy as np

NUM_TRAIN     = 49000
EPOCHS_SEARCH = 3
EPOCHS_FINAL  = 15
dtype         = torch.float32

# התיקון הקריטי מס' 1: מעבר אוטומטי ל-GPU אם קיים
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# ── 1. Augmentation ───────────────────────────────────────────────────────────
transform_train = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_eval = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

def get_loaders(batch_size=256):
    tr  = dset.CIFAR10('./intro_to_ai/datasets', train=True,
                        download=True, transform=transform_train)
    val = dset.CIFAR10('./intro_to_ai/datasets', train=True,
                        download=True, transform=transform_eval)
    te  = dset.CIFAR10('./intro_to_ai/datasets', train=False,
                        download=True, transform=transform_eval)
    return (
        DataLoader(tr,  batch_size=batch_size, num_workers=0,
                   sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN))),
        DataLoader(val, batch_size=batch_size, num_workers=0,
                   sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000))),
        DataLoader(te,  batch_size=batch_size, num_workers=0)
    )

# התיקון הקריטי מס' 2: טעינת הנתונים פעם אחת בלבד מחוץ ללולאת החיפוש!
train_loader, val_loader, test_loader = get_loaders(batch_size=256)

# ── 2. Mixup ──────────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(device) # הוספנו תמיכה ב-GPU
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(pred, ya, yb, lam):
    c = nn.CrossEntropyLoss(label_smoothing=0.1)
    return lam * c(pred, ya) + (1 - lam) * c(pred, yb)

# ── 3. CNN Architecture ───────────────────────────────────────────────────────
def build_cnn(drop_conv=0.15, drop_fc=0.4):
    def block(ic, oc, dr):
        return nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
            nn.Conv2d(oc, oc, 3, padding=1),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dr)
        )
    return nn.Sequential(
        block(3,   32,  drop_conv),   # 32×32 → 16×16
        block(32,  64,  drop_conv),   # 16×16 →  8×8
        block(64,  128, drop_conv),   #  8×8  →  4×4
        nn.Flatten(),                 # 128*4*4 = 2048
        nn.Linear(2048, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(drop_fc),
        nn.Linear(512, 10)
    )

# ── 4. Train / Eval ───────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, use_mixup):
    model.train()
    total_loss = 0.0
    bar = tqdm(loader, desc='  train', leave=False, dynamic_ncols=True)
    for x, y in bar:
        # התיקון הקריטי מס' 3: העברת הנתונים ל-GPU
        x, y = x.to(device=device, dtype=dtype), y.to(device=device, dtype=torch.long)

        optimizer.zero_grad()
        if use_mixup:
            xm, ya, yb, lam = mixup_data(x, y)
            loss = mixup_loss(model(xm), ya, yb, lam)
        else:
            loss = F.cross_entropy(model(x), y, label_smoothing=0.1)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.3f}')
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device=device, dtype=dtype), y.to(device=device, dtype=torch.long)
            correct += model(x).max(1)[1].eq(y).sum().item()
            total   += y.size(0)
    return correct / total

# ── 5. Optuna Search ──────────────────────────────────────────────────────────
def objective(trial):
    lr        = trial.suggest_float('lr',        5e-4, 3e-3, log=True)
    wd        = trial.suggest_float('wd',        1e-5, 1e-3, log=True)
    drop_conv = trial.suggest_float('drop_conv', 0.05, 0.25)
    drop_fc   = trial.suggest_float('drop_fc',   0.3,  0.5)
    use_mixup = trial.suggest_categorical('mixup', [True, False])

    # אתחול המודל והעברתו ל-GPU
    model = build_cnn(drop_conv, drop_fc).to(device)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch   = optim.lr_scheduler.OneCycleLR(
                opt, max_lr=lr,
                steps_per_epoch=len(train_loader),
                epochs=EPOCHS_SEARCH)

    ebar = trange(EPOCHS_SEARCH, desc=f'Trial {trial.number:2d}',
                  leave=False, dynamic_ncols=True)
    for _ in ebar:
        loss = train_epoch(model, train_loader, opt, sch, use_mixup)
        acc  = evaluate(model, val_loader)
        ebar.set_postfix(loss=f'{loss:.3f}', val=f'{acc*100:.1f}%')

    tqdm.write(f'  Trial {trial.number:2d} | val={acc*100:.2f}% | '
               f'lr={lr:.1e} mixup={use_mixup} '
               f'dc={drop_conv:.2f} df={drop_fc:.2f}')
    return acc

print('=' * 55)
print(f'Optuna: 10 trials × {EPOCHS_SEARCH} epochs')
print('=' * 55)

trials_bar = tqdm(total=10, desc='Optuna trials', dynamic_ncols=True)

def cb(study, trial):
    trials_bar.update(1)
    trials_bar.set_postfix(best=f'{study.best_value*100:.2f}%')

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10, callbacks=[cb])
trials_bar.close()

p = study.best_params
print(f'\nBest search val: {study.best_value*100:.2f}%')
print(f'Best params:     {p}')

# ── 6. Final Training ─────────────────────────────────────────────────────────
print(f'\n{"="*55}')
print(f'Final training: {EPOCHS_FINAL} epochs')
print(f'{"="*55}')

# אתחול מודל סופי ל-GPU
model     = build_cnn(p['drop_conv'], p['drop_fc']).to(device)
optimizer = optim.AdamW(model.parameters(), lr=p['lr'], weight_decay=p['wd'])
scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=p['lr'] * 8,
                steps_per_epoch=len(train_loader),
                epochs=EPOCHS_FINAL,
                pct_start=0.3,
                div_factor=8,
                final_div_factor=50
            )

best_val, best_weights = 0.0, None
epoch_bar = trange(EPOCHS_FINAL, desc='Training', dynamic_ncols=True)

for epoch in epoch_bar:
    avg_loss = train_epoch(model, train_loader, optimizer, scheduler, p['mixup'])
    val_acc  = evaluate(model, val_loader)
    mark     = ''

    if val_acc > best_val:
        best_val     = val_acc
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        mark         = ' ★'

    epoch_bar.set_postfix(
        val=f'{val_acc*100:.2f}%',
        best=f'{best_val*100:.2f}%'
    )
    tqdm.write(f'Epoch {epoch+1:2d}/{EPOCHS_FINAL} | '
               f'loss={avg_loss:.4f} | val={val_acc*100:.2f}%{mark}')

print(f'\n{"="*55}')
print(f'Best validation accuracy: {best_val*100:.2f}%')
print(f'{"="*55}')

model.load_state_dict(best_weights)
best_model = model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.0 MB/s eta 0:00:00
Using device: cuda


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


Optuna: 10 trials × 3 epochs


Optuna trials:   0%|          | 0/10 [00:00<?, ?it/s]

Trial  0:   0%|          | 0/3 [00:00<?, ?it/s]

...

Epoch 13/15 | loss=0.8402 | val=87.10%


  train:   0%|          | 0/192 [00:00<?, ?it/s]

Epoch 14/15 | loss=0.8252 | val=88.00%


  train:   0%|          | 0/192 [00:00<?, ?it/s]

Epoch 15/15 | loss=0.8196 | val=88.10%

Best validation accuracy: 88.10%


## Test set -- run this only once

Now that we've gotten a result we're happy with, we test our final model on the test set (which you should store in best_model). Think about how this compares to your validation set accuracy.

In [9]:
# ── Final Test Evaluation — Self Contained ────────────────────────────────────

import torch
import torch.nn as nn
import torchvision.datasets as dset
import torchvision.transforms as T
from torch.utils.data import DataLoader

dtype  = torch.float32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Best params from Optuna ───────────────────────────────────────────────────
BEST_DROP_CONV = 0.08636499344142012
BEST_DROP_FC   = 0.33668090197068673

# ── CNN Architecture (זהה לחלוטין לקוד האימון) ───────────────────────────────
def build_cnn(drop_conv=0.15, drop_fc=0.4):
    def block(ic, oc, dr):
        return nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
            nn.Conv2d(oc, oc, 3, padding=1),
            nn.BatchNorm2d(oc),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dr)
        )
    return nn.Sequential(
        block(3,   32,  drop_conv),
        block(32,  64,  drop_conv),
        block(64,  128, drop_conv),
        nn.Flatten(),
        nn.Linear(2048, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(drop_fc),
        nn.Linear(512, 10)
    )

# ── Test Loader ───────────────────────────────────────────────────────────────
transform_eval = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

cifar_test  = dset.CIFAR10('./intro_to_ai/datasets', train=False,
                            download=True, transform=transform_eval)
loader_test = DataLoader(cifar_test, batch_size=256, num_workers=0)

# ── check_accuracy_part34 (העתק מהמטלה המקורית) ──────────────────────────────
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))
    return acc

# ── טעינת המשקולות הטובות ביותר מהאימון ─────────────────────────────────────
# best_weights נשמר בתא הקודם — טוענים אותו למודל חדש
final_model = build_cnn(BEST_DROP_CONV, BEST_DROP_FC).to(device)
final_model.load_state_dict(best_weights)

# ── הרצה על ה-Test Set ────────────────────────────────────────────────────────
best_model = final_model
check_accuracy_part34(loader_test, best_model)

Checking accuracy on test set
Got 8711 / 10000 correct (87.11%)


0.8711

In [ ]:
best_model = model
check_accuracy_part34(loader_test, best_model)